# 06 – Optimierung: Gradient Descent Zoo

**Modul:** 06 – Optimierung  
**Thema:** SGD, Momentum, Adam – Implementierung & Vergleich

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)

## Verlustfunktion

Wir optimieren die Rosenbrock-Funktion (Bananenform) – ein klassisches Testproblem:

$$f(x, y) = (1-x)^2 + 100(y - x^2)^2$$

Globales Minimum bei $(1, 1)$.

In [ ]:
def rosenbrock(x, y):
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(x, y):
    dx = -2*(1 - x) - 400*x*(y - x**2)
    dy = 200*(y - x**2)
    return np.array([dx, dy])

# Optimierer implementieren
class GradientDescent:
    def __init__(self, lr=0.001):
        self.lr = lr
    
    def step(self, params, grad):
        return params - self.lr * grad

class MomentumSGD:
    def __init__(self, lr=0.001, beta=0.9):
        self.lr = lr
        self.beta = beta
        self.v = None
    
    def step(self, params, grad):
        if self.v is None:
            self.v = np.zeros_like(params)
        self.v = self.beta * self.v + (1 - self.beta) * grad
        return params - self.lr * self.v

class Adam:
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = None
        self.v = None
        self.t = 0
    
    def step(self, params, grad):
        if self.m is None:
            self.m = np.zeros_like(params)
            self.v = np.zeros_like(params)
        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1 - self.beta2) * grad**2
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)
        return params - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

def run_optimizer(optimizer, start, n_steps=1000):
    params = np.array(start, dtype=float)
    history = [params.copy()]
    losses = [rosenbrock(*params)]
    
    for _ in range(n_steps):
        grad = rosenbrock_grad(*params)
        params = optimizer.step(params, grad)
        history.append(params.copy())
        losses.append(rosenbrock(*params))
    
    return np.array(history), np.array(losses)

# Starten
start = [-1.5, 1.5]

history_sgd, losses_sgd = run_optimizer(GradientDescent(lr=0.002), start)
history_mom, losses_mom = run_optimizer(MomentumSGD(lr=0.002), start)
history_adam, losses_adam = run_optimizer(Adam(lr=0.05), start)

print(f'SGD Endverlust:      {losses_sgd[-1]:.6f}')
print(f'Momentum Endverlust: {losses_mom[-1]:.6f}')
print(f'Adam Endverlust:     {losses_adam[-1]:.6f}')

In [ ]:
# Visualisierung
x = np.linspace(-2, 2, 300)
y = np.linspace(-1, 3, 300)
X, Y = np.meshgrid(x, y)
Z = rosenbrock(X, Y)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Konturplot mit Trajektorien
axes[0].contourf(X, Y, np.log(Z + 1), levels=30, cmap='viridis', alpha=0.8)
axes[0].contour(X, Y, np.log(Z + 1), levels=30, colors='white', linewidths=0.3, alpha=0.5)

colors = ['#E74C3C', '#F39C12', '#2ECC71']
names = ['SGD', 'Momentum', 'Adam']

for hist, color, name in zip([history_sgd, history_mom, history_adam], colors, names):
    axes[0].plot(hist[:, 0], hist[:, 1], color=color, alpha=0.8, lw=2, label=name)
    axes[0].plot(hist[-1, 0], hist[-1, 1], 'o', color=color, ms=10)

axes[0].plot(1, 1, '*', color='white', ms=15, label='Minimum (1,1)')
axes[0].plot(*start, 's', color='white', ms=10, label='Start')
axes[0].set_title('Trajektorien auf Rosenbrock-Funktion')
axes[0].legend()
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')

# Konvergenzkurven
for losses, color, name in zip([losses_sgd, losses_mom, losses_adam], colors, names):
    axes[1].semilogy(losses, color=color, lw=2, label=name)
axes[1].set_title('Konvergenz (log-Skala)')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Verlust (log)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../resources/optimizer_vergleich.png', dpi=150)
plt.show()